# Optimizing Strategies in the Quacks of Quedlinburg
The boardgame _The Quacks of Quedlinburg_ features both elements of chance and player choice in its gameplay mechanics. This notebook examines data from a comprehensive simulation of the game to determine which player choices most influence the outcome and which strategies have the highest winrate overall.

## 1. Problem Definition
Given the outcome of millions of simulated matches, determine which decisions matter most in the game _The Quacks of Quedlinburg_. Furthermore, identify the strategies with the highest winrates.

## 2. Data
The dataset used in this notebook was created using a simulation coded in Python. Each mechanic of the two-player variation of the game was simulated except for the Fortune-Teller cards as the player has no influence on these cards and they apply equally to both players. The ingredient-specific rules used are as follows:
* Pumpkin (orange chip) - No special properties.
* Moth (black chip) - Standard 2-player rules.
* Spider (green chip) - For each green chip on the last or second-to-last space, receive a ruby.
* Skull (blue chip) - Draw chips equal to the value of this chip. You may play one and return the rest.
* Mushroom (red chip) - 1 or 2 orange chips already in the pot add 1 to the value of the red chip. For 3 or more oranges, add 2.
* Mandrake (yellow chip) - If the previously-played chip was white, put it back in the bag.
* Ghost (purple chip) - If you played 1 purple chip, receive 1 victory point; 2 purple chips, 1 victory point and 1 ruby; 3 or more purple chips, 2 victory points and 1 droplet move.

The simulation represented the player's strategy as a combination of elements chosen from seven key decision points:
* Which chip colors to purchase in the shop.
* Which values of chips to purchase in the shop (e.g. purchasing a 1-value chip vs a 4-value chip).
* How to spend rubies.
* When to use the flask.
* Whether or not to draw chips during the potions phase based on explosion probability.
* Whether or not to draw chips during the potions phase based on the current round.
* Whether to choose money or victory points in the event of an explosion.

We refer to each unique combination of choices as a strategy profile. The total number of strategy profiles generated amounted to 914,400. Each strategy profile was played 100 times against a player making random choices.

## 3. Evaluation
The performance of each strategy profile will be compared using its Chess Win Rate(CWR), calculated as follows:
CWR = (Wins + (0.5 * Draws)) / Total Games
This metric was selected so draws can be accurately factored into the winrate.

To determine a baseline CWR, 10,000 games were simulated in which both players used a random strategy profile. The results were as follows: 4720 wins, 4798 losses and 482 draws. This results in a CWR of 49.61%, which aligns with our expectation that wins and losses should be approximately equal when a strategy is used against itself.

We will also rank the impact of each decision point using feature importance generated by a random forest regressor.

Finally, we will determine how much each option contributes to the overall victory points gained using a linear regression model.

## 4. Features
The dataset consists of the following features:
* Color strategy:
    - color_strat_all - a unique ID for the particular combination of colors. There are 7 different colors to choose from: green, blue, red, yellow, orange, purple, and black. All combinations of these colors in all selection sizes (i.e. 1, 2, 3, 4, 5, 6, and 7) amount to 127 (with IDs ranging from 1 to 127).
    - color_strat_green - whether or not green was part of the color strategy (1=yes, 0=no).
    - color_strat_blue - whether or not blue was part of the color strategy (1=yes, 0=no).
    - color_strat_red - whether or not red was part of the color strategy (1=yes, 0=no).
    - color_strat_yellow - whether or not yellow was part of the color strategy (1=yes, 0=no).
    - color_strat_orange - whether or not orange was part of the color strategy (1=yes, 0=no).
    - color_strat_purple - whether or not purple was part of the color strategy (1=yes, 0=no).
    - color_strat_black - whether or not black was part of the color strategy (1=yes, 0=no).

* Value strategy:
    - value_strat_pair - whether or not the "highest value pair" strategy was used (1=yes, 0=no). The player attempts to purchase the most expensive chips they can afford.
    - value_strat_single - whether or not the "highest single value" strategy was used (1=yes, 0=no). The player attempts to buy the most expensive expensive chip they can afford even if it means only buying one chip instead of two.
    - value_strat_disparity - whether or not the "least disparity" strategy was used (1=yes, 0=no). The player prefers buying two mid-value chips over one high-value and one low-value chip.  

* Ruby strategy:
    - ruby_strat_save - whether or not the "save" strategy was used (1=yes, 0=no). The player saves their rubies to trade for victory points at the end of the game.
    - ruby_strat_droplet - whether or not the "droplet" strategy was used (1=yes, 0=no). The player always buys droplet advances with their rubies.
    - ruby_strat_flask - whether or not the "flask" strategy was used (1=yes, 0=no). The player always buys a flask refresh with their rubies.
    - ruby_strat_balanced - whether or not the "balanced" strategy was used (1=yes, 0=no). The player buys both droplet advances and flask refreshes with their rubies.

* Flask strategy:
    - flask_strat_three - whether or not the "always three" strategy was used (1=yes, 0=no). The player always uses their flask when a white-3 is drawn.
    - flask_strat_risk - whether or not the "at risk" strategy was used (1=yes, 0=no). The player uses their flask as soon as their is a risk of exploding.
    - flask_strat_ruby - whether or not the "no ruby" strategy was used (1=yes, 0=no). The player uses their flask if they won't receive a ruby.
    - flask_strat_seventy - whether or not the "seventy" strategy was used (1=yes, 0=no). The player uses their flask if there is a risk of exploding, but at least 70% of their non-white chips have not yet been played.
    - flask_strat_fifty - whether or not the "fifty" strategy was used (1=yes, 0=no). The player uses their flask if there is a risk of exploding, but at least 50% of their non-white chips have not yet been payed.

* Explosion risk tolerance probability-based strategy:
    - ex_prob_strat_high - whether or not the "high" strategy was used (1=yes, 0=no). The player continues drawing chips even if there is up to an 80% probability they will explode.
    - ex_prob_strat_medium - whether or not the "medium" strategy was used (1=yes, 0=no). The player continues drawing chips even if there is up to a 60% probability they will explode.
    - ex_prob_strat_low - whether or not the "low" strategy was used (1=yes, 0=no). The player continues drawing chips even if there is ais up to a 40% probability they will explode.
    - ex_prob_strat_very_low - whether or not the "very low" strategy was used (1=yes, 0=no). The player continues drawing chips even if there is is up to a 20% probability they will explode.
    - ex_prob_strat_never - whether or not the "never" strategy was used (1=yes, 0=no). The player continues drawing chips only if there is a 0% probability they will explode.

* Explosion risk tolerance round-based strategy:
    - ex_round_strat_always - whether or not the "always" strategy was used (1=yes, 0=no). The player risks exploding during all 9 rounds.
    - ex_round_strat_early - whether or not the "early" strategy was used (1=yes, 0=no). The player risks exploding during the first 3 rounds.
    - ex_round_strat_early_mid - whether or not the "early-mid" strategy was used (1=yes, 0=no). The player risks exploding during the first 6 rounds.
    - ex_round_strat_mid - whether or not the "mid" strategy was used (1=yes, 0=no). The player risks exploding during the middle 3 rounds.
    - ex_round_strat_mid_late - whether or not the "mid-late" strategy was used (1=yes, 0=no). The player risks exploding during the last 6 rounds.
    - ex_round_strat_late - whether or not the "late" strategy was used (1=yes, 0=no). The player risks exploding during the last 3 rounds.

* Exploded strategy:
    - exploded_strat_points - whether or not the "points" strategy was used (1=yes, 0=no). The player always chooses victory points if they have exploded.
    - exploded_strat_money - whether or not the "money" strategy was used (1=yes, 0=no). The player always chooses money if they have exploded.
    - exploded_strat_early - whether or not the "round-based early" strategy was used (1=yes, 0=no). The player chooses money if the current round is 3 or less.
    - exploded_strat_mid - whether or not the "round-based late" strategy was used (1=yes, 0=no). The player chooses money if the current round is 6 or less.

* Chip distribution:
    - green-1 - How many green-1 chips the player owned by the 9th round.
    - green-2 - How many green-2 chips the player owned by the 9th round.
    - green-4 - How many green-4 chips the player owned by the 9th round.
    - blue-1 - How many blue-1 chips the player owned by the 9th round.
    - blue-2 - How many blue-2 chips the player owned by the 9th round.
    - blue-4 - How many blue-4 chips the player owned by the 9th round.
    - red-1 - How many red-1 chips the player owned by the 9th round.
    - red-2 - How many red-2 chips the player owned by the 9th round.
    - red-4 - How many red-4 chips the player owned by the 9th round.
    - yellow-1 - How many yellow-1 chips the player owned by the 9th round.
    - yellow-2 - How many yellow-2 chips the player owned by the 9th round.
    - yellow-4 - How many yellow-4 chips the player owned by the 9th round.
    - orange-1 - How many orange-1 chips the player owned by the 9th round.
    - purple-1 - How many purple-1 chips the player owned by the 9th round.
    - black-1 - How many black-1 chips the player owned by the 9th round.

* Color distribution:
    - green_count - How many green chips the player owned by the 9th round.
    - blue_count - How many blue chips the player owned by the 9th round.
    - red_count - How many red chips the player owned by the 9th round.
    - yellow_count - How many yellow chips the player owned by the 9th round.
    - orange_count - How many orange chips the player owned by the 9th round.
    - purple_count - How many purple chips the player owned by the 9th round.
    - black_count - How many black chips the player owned by the 9th round.

* Value distribution:
    - one_count - How many chips with a value of 1 the player owned by the 9th round.
    - two_count - How many chips with a value of 2 the player owned by the 9th round.
    - four_count - How many chips with a value of 4 the player owned by the 9th round.

* Milestones:
    - final_droplet_position - How far the player moved their droplet by the end of the game.
    - farthest_space_reached - The farthest space the player reached in any round.
    - explosion_count - How many times the player exploded by the end of the game.

* Outcome:
    - player_a_vp - How many victory points player A (the strategy-employing player) scored by the end of the game.
    - player_b_vp - How many victory points player B (the random player) scored by the end of the game.
    - margin - The difference in victory points between player A and player B.
    - percent_dif - The percent difference in victory points between player A and player B.
    - tie - Whether or not the game was a tie (1=yes, 0=no).
    - player_a_won - Whether or not player A (the strategy-employing player) won the game or not (1=yes, 0=no).
* run_id - A unique number identifying the match.

## 5. Preparation
Load the libraries and the data.

In [1]:
# libraries
import polars as pl
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

In [2]:
strategy_features = [
    # Colors
    "color_strat_green", "color_strat_blue", "color_strat_red", "color_strat_yellow",
    "color_strat_orange", "color_strat_purple", "color_strat_black",
    # Value
    "value_strat_pair", "value_strat_single", "value_strat_disparity",
    # Ruby
    "ruby_strat_save", "ruby_strat_droplet", "ruby_strat_flask", "ruby_strat_balanced",
    # Flask
    "flask_strat_three", "flask_strat_risk", "flask_strat_ruby", "flask_strat_seventy", "flask_strat_fifty",
    # Explosion probability
    "ex_prob_strat_high", "ex_prob_strat_medium", "ex_prob_strat_low", "ex_prob_strat_very_low", "ex_prob_strat_never",
    # Explosion round
    "ex_round_strat_always", "ex_round_strat_early", "ex_round_strat_early_mid",
    "ex_round_strat_mid", "ex_round_strat_mid_late", "ex_round_strat_late",
    # Exploded
    "exploded_strat_points", "exploded_strat_money", "exploded_strat_early", "exploded_strat_mid"
]

# Read parquet files
path_to_parquet_files: str = r"C:\Users\Daniel\PycharmProjects\Quacks_Simulator\simulation_output\full_simulation_results"
lazy_df = pl.scan_parquet(path_to_parquet_files + r"\*.parquet")

# Aggregate by strategy features
print("Aggregating by strategy combinations...")
aggregated_df = (
    lazy_df
    .group_by(strategy_features)
    .agg([
        ((pl.col("player_a_won").sum() + (0.5 * pl.col("tie").sum())) / pl.col("player_a_won").count()).alias("chess_win_rate"),
        pl.col("margin").mean().alias("avg_margin")
    ])
    .collect(engine="streaming")
)
print("Aggregation complete!")

Aggregating by strategy combinations...
Aggregation complete!


## 6. Analysis
We will use two machine learning models to analyze the aggregated data. The random forest regression model will illuminate via feature importance which strategic categories mattered most. The linear regressor will show how much each strategy option contributed to victory points, the key objective for winning the game.

In [4]:
# Random Forest Regressor
df_analysis = aggregated_df.to_pandas()

X = df_analysis[strategy_features]
y = df_analysis["chess_win_rate"]

# Train the model
rf = RandomForestRegressor(n_estimators=50, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=strategy_features)

category_importance = {
    "Color Selection": importances[importances.index.str.startswith("color_")].sum(),
    "Value Selection": importances[importances.index.str.startswith("value_")].sum(),
    "Ruby Purchasing": importances[importances.index.str.startswith("ruby_")].sum(),
    "Flask Usage": importances[importances.index.str.startswith("flask_")].sum(),
    "Risk Tolerance (Probability)": importances[importances.index.str.startswith("ex_prob_")].sum(),
    "Risk Tolerance (Round)": importances[importances.index.str.startswith("ex_round_")].sum(),
    "Exploded Strategy": importances[importances.index.str.startswith("exploded_")].sum(),
}

print("STRATEGIC DECISION HIERARCHY")
print("----------------------------")
for cat, imp in sorted(category_importance.items(), key=lambda x: x[1], reverse=True):
    print(f"{cat}: {imp*100:.2f}% of overall impact")

STRATEGIC DECISION HIERARCHY
Risk Tolerance (Probability): 45.83% of overall impact
Risk Tolerance (Round): 23.06% of overall impact
Color Selection: 11.20% of overall impact
Exploded Strategy: 11.14% of overall impact
Ruby Purchasing: 5.64% of overall impact
Value Selection: 2.08% of overall impact
Flask Usage: 1.04% of overall impact


In [5]:
# Linear Regressor
y_margin = df_analysis["avg_margin"]

lr = LinearRegression()
lr.fit(X, y_margin)

coefficients = pd.Series(lr.coef_, index=strategy_features)

In [25]:
sorted_coefficients = coefficients.sort_values(ascending=False)

categories = {
    "Color Selection": "color_strat_",
    "Value Selection": "value_strat_",
    "Ruby Purchasing": "ruby_strat_",
    "Flask Usage": "flask_strat_",
    "Risk Tolerance (Prob)": "ex_prob_strat",
    "Risk Tolerance (Round)": "ex_round_strat",
    "Exploded Strategy": "exploded_strat_"
}

print("All Strategy Choices Ranked")
print("---------------------------")
print(sorted_coefficients)
print()

for display_name, prefix in categories.items():
    print(display_name.center(30, "-"))
    category_coefs = sorted_coefficients[sorted_coefficients.index.str.startswith(prefix)]
    category_coefs.index = category_coefs.index.str.replace(prefix, "")

    for option, val in category_coefs.items():
        sign = "+" if val >= 0 else ""
        print(f" * {option:<15} : {sign}{val:.2f} VPs")
    print()

All Strategy Choices Ranked
---------------------------
ex_prob_strat_very_low      2.699059
ex_round_strat_early        1.406673
color_strat_black           1.261545
ex_prob_strat_low           1.154384
ex_prob_strat_never         1.138929
color_strat_purple          1.130873
ruby_strat_droplet          0.948952
exploded_strat_mid          0.872675
exploded_strat_early        0.547751
flask_strat_seventy         0.544798
color_strat_blue            0.443410
flask_strat_fifty           0.401015
color_strat_orange          0.366377
ex_round_strat_late         0.353770
ruby_strat_balanced         0.202237
ex_round_strat_early_mid    0.187413
color_strat_red             0.171988
value_strat_pair            0.148199
value_strat_disparity       0.134431
ruby_strat_save             0.095755
color_strat_green           0.010589
ex_round_strat_mid         -0.074820
flask_strat_risk           -0.114414
value_strat_single         -0.282630
flask_strat_ruby           -0.414402
flask_strat_three  

## 7. Strategy Profiles Ranked
Using the chess win rate as our metric, we rank the 25 best and worst strategy profiles.

In [34]:
sorted_profiles = df_analysis.sort_values(by="chess_win_rate", ascending=False)
best_profiles = sorted_profiles.head(25)
worst_profiles = sorted_profiles.tail(25).sort_values(by="chess_win_rate", ascending=True)

print("======================")
print("BEST STRATEGY PROFILES")
print("======================\n")

for rank, (index, row) in enumerate(best_profiles.iterrows()):
    print(f"#{rank + 1} STRATEGY PROFILE (Win Rate: {row['chess_win_rate']*100:.2f}%):")
    for feature in strategy_features:
        if row[feature] == 1:
            print(f" - {feature}")
    print()

print("=======================")
print("WORST STRATEGY PROFILES")
print("=======================\n")

for rank, (index, row) in enumerate(worst_profiles.iterrows()):
    print(f"#{rank + 1} STRATEGY PROFILE (Win Rate: {row['chess_win_rate']*100:.2f}%):")
    for feature in strategy_features:
        if row[feature] == 1:
            print(f" - {feature}")
    print()

BEST STRATEGY PROFILES

#1 STRATEGY PROFILE (Win Rate: 96.50%):
 - color_strat_blue
 - color_strat_orange
 - color_strat_purple
 - color_strat_black
 - value_strat_single
 - ruby_strat_balanced
 - flask_strat_three
 - ex_prob_strat_very_low
 - ex_round_strat_always
 - exploded_strat_mid

#2 STRATEGY PROFILE (Win Rate: 96.00%):
 - color_strat_blue
 - color_strat_purple
 - color_strat_black
 - value_strat_pair
 - ruby_strat_droplet
 - flask_strat_seventy
 - ex_prob_strat_very_low
 - ex_round_strat_always
 - exploded_strat_mid

#3 STRATEGY PROFILE (Win Rate: 96.00%):
 - color_strat_blue
 - color_strat_orange
 - color_strat_purple
 - color_strat_black
 - value_strat_single
 - ruby_strat_droplet
 - flask_strat_three
 - ex_prob_strat_low
 - ex_round_strat_early_mid
 - exploded_strat_mid

#4 STRATEGY PROFILE (Win Rate: 96.00%):
 - color_strat_blue
 - color_strat_red
 - color_strat_yellow
 - color_strat_orange
 - color_strat_black
 - value_strat_disparity
 - ruby_strat_droplet
 - flask_strat_s

## 8. Conclusions
In this section, we draw conclusions based on the results of our analysis.

#### Impact of Decision Points
Our random forest regressor determined which decision points had the greatest impact on winning the game using feature importance:

- Risk Tolerance (Probability): 45.83% of overall impact
- Risk Tolerance (Round): 23.06% of overall impact
- Color Selection: 11.20% of overall impact
- Exploded Strategy: 11.14% of overall impact
- Ruby Purchasing: 5.64% of overall impact
- Value Selection: 2.08% of overall impact
- Flask Usage: 1.04% of overall impact

We can see that nearly 70% of the outcome of the game is determined by how risk is managed, i.e., how does the player respond to the ever-increasing risk of exploding their pot during the potions phase. The probability of exploding bore the most weight, nearly twice as much as the current round. Surprisingly, the color selection plays a much smaller role overall. Of nearly equal importance is how the player responds to an explosion, which forces a choice between victory points or money. Neither how rubies are spent nor which values of chips are purchased plays a signficant role; however, quite surprising is that flask usage is nearly negligible. This further implies that purchasing flask refreshes with rubies is likely a waste.

#### Impact of Individual Strategy Options
Using linear regression, we see how many victory points each strategic option contributed to the average margin between the players:

----Risk Tolerance (Prob)-----
 * _very_low       : +2.70 VPs
 * _low            : +1.15 VPs
 * _never          : +1.14 VPs
 * _medium         : -1.37 VPs
 * _high           : -3.63 VPs

We can see in the chart above that the most effective strategies for handling risk are the safer options, with "very low" scoring the best. This is significant because _The Quacks of Quedlinburg_ is a "push-your-luck" style game - players take risk in the hopes of garnering greater rewards. The data show that _some_ level of risk-taking is helpful, but not too much. The "very low" strategy option dictates that a player should push their luck if the probability of pulling an explosion-causing chip is 20% or less. For players who wish to avoid the math, the "never" strategy might be most suitable as it still scored positively. Both the "medium" and "high" risk strategies result in losing points on average.


----Risk Tolerance (Round)----
 * _early          : +1.41 VPs
 * _late           : +0.35 VPs
 * _early_mid      : +0.19 VPs
 * _mid            : -0.07 VPs
 * _always         : -0.85 VPs
 * _mid_late       : -1.02 VPs

The round also plays a role in determining if a player should push their luck. The most effective strategy is to risk exploding during the "early" rounds, i.e., 1-3. Suprisingly, risking exploding "late" (the last three rounds) also resulted in a positive contribution to victory points. Interestingly, 3 of the 4 worst options consisted of being willing to explode on 6 rounds rather than 3. This coincides with the above conlusion that a more conservative playstyle tends to be more effective.


-------Color Selection--------
 * black           : +1.26 VPs
 * purple          : +1.13 VPs
 * blue            : +0.44 VPs
 * orange          : +0.37 VPs
 * red             : +0.17 VPs
 * green           : +0.01 VPs
 * yellow          : -0.82 VPs

Nearly all colors contribute to scoring victory points, but another suprising result emerged: yellow chips actually _decrease_ victory points on average. The yellow chip special ability seems powerful on the surface - if the previous chip played was white, remove that chip. Perhaps this effect is unlikely to trigger often enough to justify the high expense of these chips. Green chips also disappoint - while they are cheap relative to their values, perhaps their special ability (granting extra rubies) simply doesn't contribute enough to victory since rubies themselves - as we have seen - have a low impact. Furthermore, the trigger for the green chip is also not likely to happen. Interestingly, 3 of the 4 best chip colors can only be purchased in a value of 1.

   
------Exploded Strategy-------
 * mid             : +0.87 VPs
 * early           : +0.55 VPs
 * points          : -0.59 VPs
 * money           : -0.83 VPs

Neither rigid strategy (only points or only money) scored well. The best strategy ("mid") is to choose money during the first 6 rounds.


-------Ruby Purchasing--------
 * droplet         : +0.95 VPs
 * balanced        : +0.20 VPs
 * save            : +0.10 VPs
 * flask           : -1.25 VPs

We saw earlier that the flask strategy has the least impact in winning the game. Therefore, it is no suprise that purchasing flask refreshes with rubies is a waste of money. Saving rubies to trade in for victory points, however, is also a poor strategy. The best strategy is to purchase droplet advances.


-------Value Selection--------
 * pair            : +0.15 VPs
 * disparity       : +0.13 VPs
 * single          : -0.28 VPs

The best strategy, "highest value pair," seeks to purchase the most expensive chips possible with the money available. Purchasing a high value chip at the expense of purchasing a second chip ("highest single value") negatively impacts the score.


---------Flask Usage----------
 * seventy         : +0.54 VPs
 * fifty           : +0.40 VPs
 * risk            : -0.11 VPs
 * ruby            : -0.41 VPs
 * three           : -0.42 VPs

The only good uses of the flask are to save the player from the most unlucky situations: having many colored chips still in the bag when faced with the risk of exploding. In this case, the player should flask in an attempt to access more of those chips. This does require some math, however, with the best strategy employing the flask when 70% of colored chips remain.

#### The Best Strategy Profile
This is the strategy profile with the highest winrate (96.5%):
 - colors: blue, orange, purple, black
 - value: highest single value
 - ruby: balanced
 - flask: three
 - explosion risk tolerance (probability-based): very low
 - explosion risk tolerance (round-based): always
 - exploded: mid

We examined general principles of best play above. However, the interaction of the strategic options may still result in a violation of the "best" practices. This plays out in the highest-scoring strategy profile. The value, ruby, flask, and round-based risk tolerance strategies are all sub-optimal according to our analysis. Playing "highest single value" may work here because of the presence of the orange chips in the color strategy. Orange chips are the least expensive, which would mean that a pair would still be purchased in many cases. Furthermore, the blue chips benefit greatly from having higher values as this allows the player to draw more chips at once when choosing the next chip to play. Having several blue chips can also result in a "chain reaction" in which the blue chip is played and another blue chip is drawn, allowing the player to again draw several chips and choose one to play. This may reduce the likelihood, of facing explosion risk with still 50-70% of chips unplayed, rendering the "fifty" and "seventy" flask strategies as less effective than usual. The blue chips may also be the reason that playing with a round-based explosion risk tolerance strategy of "always" is less risky.

Since the best strategy profile is of particular interest, I ran it in a custom simulation consisting of 10,000 games vs a random player. It achieved 8556 wins, 1181 losses, and 263 draws which amounts to a more modest, yet respectable winrate of 86.9%.


As a further comparison, I ran a custom simulation using the best individual options as analyzed above, resulting in 7882 wins, 1779 losses, and 339 draws with a winrate of 80.5%. Thus, the counter-intuitive best strategy profile further proves its worth.


## 9. Further Research
The conflict between the best strategy profile above and the best options indicates that there may be some hidden interactions between the gameplay mechanics. A deeper understanding of these could be achieved by analyzing feature interaction. This could uncover interesting and fun alternative approaches to winning the game.

We also saw a nearly 10% difference in the winrate achieved by the best strategy profile in the full simulation vs the custom simulation. It is clear our analysis would benefit from a simulation consisting of more trials per strategy profile. This would, however, require significantly more compute.

A third area of further research is related to how the simulation itself is coded. The simulation built for this analysis used various human-designed strategies that were simply tested in different combinations. However, another approach would be for the simulation to make purely random choices and to log the state of the game at each juncture. A machine learning model could than be trained on this data allowing strategies to emerge organically. In this way, it may be possible to discover strategies that are counter-intuitive from a human standpoint.